# Saba Yisroel Whisper Fine-Tuning
Fine-tune on 55 min of high-confidence audio.

NNNNM!

In [ ]:
!pip install -q transformers datasets accelerate evaluate jiwer soundfile librosa
print("Done!")

In [ ]:
import urllib.request, json
for f in ["saba_training_pairs.json","saba_lexicon.json","saba_vocabulary.txt"]:
    urllib.request.urlretrieve(f"https://ajew.org/data/saba-training/{f}", f)
    print(f"Downloaded {f}")
with open("saba_training_pairs.json") as fh:
    pairs = json.load(fh)
print(f"Training segments: {len(pairs)}")

In [ ]:
# Step 3: Download pre-extracted training audio (no Drive needed!)
import urllib.request, zipfile, os

# Try downloading from ajew.org
AUDIO_URL = "https://github.com/petteknanach/ajew-org/releases/download/saba-training-v1/saba_training_audio.zip"
try:
    print("Downloading training audio from ajew.org...")
    urllib.request.urlretrieve(AUDIO_URL, "saba_training_audio.zip")
    print("Downloaded! Extracting...")
except:
    print("Download failed. Upload saba_training_audio.zip manually:")
    from google.colab import files
    uploaded = files.upload()

with zipfile.ZipFile("saba_training_audio.zip", "r") as z:
    z.extractall("audio")
print(f"Extracted {len(os.listdir('audio'))} files")
RECORDINGS = "audio"

In [ ]:
# Step 4: Load pre-extracted segments
import librosa, numpy as np, json, os

with open(os.path.join(RECORDINGS, "manifest.json")) as f:
    manifest = json.load(f)

segments = []
for i, m in enumerate(manifest):
    path = os.path.join(RECORDINGS, m["file"])
    if os.path.exists(path):
        audio, sr = librosa.load(path, sr=16000)
        segments.append({"aud": audio, "txt": m["text"]})
    if (i+1) % 50 == 0:
        print(f"Loaded {i+1}/{len(manifest)}")
total_sec = sum(len(s["aud"])/16000 for s in segments)
print(f"Loaded: {len(segments)} segments, {total_sec/60:.1f} min")

In [ ]:
from datasets import Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration

dataset = Dataset.from_dict({"audio": [s["aud"] for s in segments], "text": [s["txt"] for s in segments]})
split = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(split['train'])}, Eval: {len(split['test'])}")

processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="he", task="transcribe")
model.config.suppress_tokens = []
print("Model loaded!")

In [ ]:
def prepare_dataset(batch):
    input_features = processor(batch["audio"], sampling_rate=16000, return_tensors="np").input_features[0]
    batch["input_features"] = input_features
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_ds = split["train"].map(prepare_dataset, remove_columns=["audio","text"])
eval_ds = split["test"].map(prepare_dataset, remove_columns=["audio","text"])
print("Features ready!")

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any
    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == model.config.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

collator = DataCollatorSpeechSeq2Seq(processor=processor)
print("Collator ready!")

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate

metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100 * metric.compute(predictions=pred_str, references=label_str)}

args = Seq2SeqTrainingArguments(
    output_dir="./saba-whisper",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    fp16=True,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=args, model=model,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collator, compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
print("Ready to train!")

In [ ]:
print("NNNNM! Training Saba-tuned Whisper...")
trainer.train()
print("Training complete!")

In [ ]:
trainer.save_model("./saba-whisper")
processor.save_pretrained("./saba-whisper")
import shutil
shutil.copytree("./saba-whisper", "/content/drive/MyDrive/saba-whisper-finetuned", dirs_exist_ok=True)
print("Model saved to Google Drive!")